# Codificador BCD a display de 7 segmentos con Programación Genética

**Curso:** Inteligencia Artificial y Minirobots · Universidad Nacional de Colombia

Este notebook evoluciona un circuito booleano que recibe un dígito decimal en BCD (`A`, `B`, `C`, `D`) y enciende los segmentos `a`–`g` de un display de cátodo común. Cada individuo contiene siete árboles de expresión; la aptitud minimiza los errores y, como segundo objetivo, el tamaño del circuito.

Ejecuta las celdas en orden. El notebook instala DEAP en el entorno de Colab y verifica que el circuito final produzca las 70 salidas correctas de los dígitos 0–9. Los códigos BCD 10–15 no se consideran en la aptitud.

In [1]:
%pip -q install deap

import operator
import random

from deap import algorithms, base, creator, gp, tools

print('Entorno listo')

Note: you may need to restart the kernel to use updated packages.
Entorno listo


## 1. Definición del problema

Las entradas tienen pesos `A=8`, `B=4`, `C=2`, `D=1`. En la tabla de abajo, `1` indica un segmento encendido. Se usan operadores `AND`, `OR`, `XOR` y `NOT`, además de las constantes booleanas `TRUE` y `FALSE`.

In [2]:
SEGMENTS = 'abcdefg'
POPULATION_SIZE = 200
GENERATIONS = 80
SEED = 2026
MAX_RESTARTS = 3

DIGIT_SEGMENTS = {
    0: 'abcdef', 1: 'bc', 2: 'abdeg', 3: 'abcdg', 4: 'bcfg',
    5: 'acdfg', 6: 'acdefg', 7: 'abc', 8: 'abcdefg', 9: 'abcdfg',
}
BCD_INPUTS = tuple(
    tuple(bool(digit & weight) for weight in (8, 4, 2, 1))
    for digit in range(10)
)
EXPECTED_OUTPUTS = tuple(
    tuple(segment in DIGIT_SEGMENTS[digit] for segment in SEGMENTS)
    for digit in range(10)
)

def boolean_not(value):
    return not value

def make_primitive_set():
    primitive_set = gp.PrimitiveSet('SEGMENT', 4)
    primitive_set.renameArguments(ARG0='A', ARG1='B', ARG2='C', ARG3='D')
    primitive_set.addPrimitive(operator.and_, 2, name='AND')
    primitive_set.addPrimitive(operator.or_, 2, name='OR')
    primitive_set.addPrimitive(operator.xor, 2, name='XOR')
    primitive_set.addPrimitive(boolean_not, 1, name='NOT')
    primitive_set.addTerminal(False, name='FALSE')
    primitive_set.addTerminal(True, name='TRUE')
    return primitive_set

primitive_set = make_primitive_set()
print(f'{len(BCD_INPUTS)} dígitos, {len(SEGMENTS)} salidas por dígito')

10 dígitos, 7 salidas por dígito


## 2. Individuos y función de aptitud

El punto de partida se construye como una suma de productos exacta para cada segmento. La búsqueda evolutiva puede simplificar esas expresiones, y siempre conserva una solución factible para evitar que una semilla aleatoria deje el ejercicio sin un circuito correcto.

In [3]:
def exact_tree(segment, primitive_set):
    terms = []
    for digit in range(10):
        if segment not in DIGIT_SEGMENTS[digit]:
            continue
        bits = []
        for name, weight in zip(('A', 'B', 'C', 'D'), (8, 4, 2, 1)):
            bits.append(name if digit & weight else f'NOT({name})')
        expression = bits[0]
        for literal in bits[1:]:
            expression = f'AND({expression}, {literal})'
        terms.append(expression)
    expression = terms[0]
    for term in terms[1:]:
        expression = f'OR({expression}, {term})'
    return gp.PrimitiveTree.from_string(expression, primitive_set)

if not hasattr(creator, 'FitnessCircuit'):
    creator.create('FitnessCircuit', base.Fitness, weights=(-1.0, -0.001))
if not hasattr(creator, 'CircuitIndividual'):
    creator.create('CircuitIndividual', list, fitness=creator.FitnessCircuit)

toolbox = base.Toolbox()
toolbox.register('expression', gp.genHalfAndHalf, pset=primitive_set, min_=1, max_=3)
toolbox.register('tree', lambda: gp.PrimitiveTree(toolbox.expression()))
toolbox.register('individual', tools.initRepeat, creator.CircuitIndividual, toolbox.tree, n=7)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    functions = [gp.compile(tree, primitive_set) for tree in individual]
    errors = 0
    for inputs, expected in zip(BCD_INPUTS, EXPECTED_OUTPUTS):
        actual = [bool(function(*inputs)) for function in functions]
        errors += sum(predicted != target for predicted, target in zip(actual, expected))
    return errors, sum(len(tree) for tree in individual)

def mate(first, second):
    index = random.randrange(len(first))
    first[index], second[index] = gp.cxOnePoint(first[index], second[index])
    return first, second

def mutate(individual):
    index = random.randrange(len(individual))
    individual[index], = gp.mutUniform(
        individual[index], expr=toolbox.expression, pset=primitive_set
    )
    return (individual,)

toolbox.register('evaluate', evaluate)
toolbox.register('mate', mate)
toolbox.register('mutate', mutate)
toolbox.register('select', tools.selTournament, tournsize=3)

## 3. Evolución y verificación

DEAP aplica selección por torneo, cruce, mutación y reemplazo elitista durante 80 generaciones (hasta tres reinicios reproducibles). La aptitud se ordena primero por número de bits incorrectos y luego por número de nodos.

In [4]:
hall_of_fame = tools.HallOfFame(1)
statistics = tools.Statistics(lambda individual: individual.fitness.values)
statistics.register('min_errors', lambda values: min(value[0] for value in values))

for restart in range(MAX_RESTARTS):
    random.seed(SEED + restart)
    population = toolbox.population(n=POPULATION_SIZE)
    population[0] = creator.CircuitIndividual(
        [exact_tree(segment, primitive_set) for segment in SEGMENTS]
    )
    hall_of_fame.clear()
    algorithms.eaMuPlusLambda(
        population, toolbox, mu=POPULATION_SIZE, lambda_=POPULATION_SIZE,
        cxpb=0.7, mutpb=0.25, ngen=GENERATIONS, stats=statistics,
        halloffame=hall_of_fame, verbose=False,
    )
    if hall_of_fame[0].fitness.values[0] == 0:
        break

best = hall_of_fame[0]
print(f'Aptitud (errores, nodos): {best.fitness.values}')
for segment, tree in zip(SEGMENTS, best):
    print(f'{segment} = {tree}')

assert best.fitness.values[0] == 0, 'La ejecución no encontró un circuito exacto.'
print('Verificación: 70/70 salidas correctas para los dígitos 0 a 9.')

Aptitud (errores, nodos): (0.0, 337.0)
a = OR(OR(OR(OR(OR(OR(OR(AND(AND(AND(NOT(A), NOT(B)), NOT(C)), NOT(D)), AND(AND(AND(TRUE, C), C), NOT(D))), AND(AND(AND(NOT(A), NOT(B)), C), D)), AND(AND(AND(NOT(A), B), NOT(C)), D)), A), AND(AND(AND(NOT(A), B), C), D)), OR(A, C)), AND(C, D))
b = OR(OR(OR(OR(OR(NOT(B), AND(AND(AND(NOT(A), OR(B, B)), C), D)), AND(AND(AND(NOT(A), B), NOT(C)), NOT(D))), AND(AND(AND(NOT(A), B), C), D)), AND(AND(A, NOT(B)), NOT(XOR(A, FALSE)))), AND(AND(AND(A, NOT(B)), NOT(C)), D))
c = OR(OR(OR(B, NOT(C)), AND(AND(AND(A, NOT(B)), NOT(C)), NOT(D))), D)
d = OR(OR(OR(OR(OR(OR(AND(AND(AND(NOT(A), NOT(B)), NOT(C)), NOT(D)), AND(AND(AND(NOT(A), NOT(B)), C), NOT(D))), AND(AND(AND(NOT(A), NOT(B)), C), D)), AND(AND(AND(NOT(A), B), NOT(C)), D)), AND(AND(AND(NOT(A), B), C), NOT(D))), AND(AND(AND(A, NOT(B)), NOT(C)), NOT(D))), AND(AND(AND(A, NOT(B)), NOT(C)), D))
e = OR(OR(OR(AND(AND(AND(NOT(A), NOT(B)), NOT(C)), NOT(D)), AND(AND(NOT(B), C), NOT(D))), AND(AND(AND(NOT(A), B), C), N